# Sistema de Análisis sísmico - Volcán Deception
## Predicción con LSTM e IA

Análisis de patrones sísmicos y predicción de magnitudes usando redes neuronales LSTM

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar estilo
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Agregar rutas
sys.path.insert(0, '/home/cami/Desktop/Software3/TradingAlgoritmico')

import config
from src.data_loader import SeismicDataLoader
from src.model import SeismicLSTM, SeismicAnomalyDetector

print("✓ Librerías cargadas exitosamente")

## 1. Exploración de Datos

In [ ]:
# Generar datos sísmicos sintéticos
loader = SeismicDataLoader()
data = loader.generate_sample_data(365)  # 1 año de datos

print(f"Dataset shape: {data.shape}")
print(f"\nPrimeros registros:")
print(data.head())
print(f"\nEstadísticas:")
print(data[['magnitude', 'depth', 'latitude', 'longitude']].describe())

In [ ]:
# Visualizar distribución de magnitudes
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma de magnitudes
axes[0].hist(data['magnitude'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Magnitud')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Magnitudes')
axes[0].grid(True, alpha=0.3)

# Eventos a lo largo del tiempo
data_sorted = data.sort_values('time')
axes[1].scatter(range(len(data_sorted)), data_sorted['magnitude'], alpha=0.6, s=30)
axes[1].set_xlabel('Días')
axes[1].set_ylabel('Magnitud')
axes[1].set_title('Series Temporal de Magnitudes')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Magnitud promedio: {data['magnitude'].mean():.2f}")
print(f"Magnitud máxima: {data['magnitude'].max():.2f}")
print(f"Profundidad promedio: {data['depth'].mean():.2f} km")

## 2. Preparación de Datos

In [ ]:
# Filtrar por magnitud mínima
loader.filter_by_magnitude(config.MIN_MAGNITUDE)
print(f"Eventos después de filtrado: {len(loader.data)}")

# Normalizar características
feature_cols = ['magnitude', 'depth']
loader.normalize_features(feature_cols, method='minmax')

print(f"\nCaracterísticas normalizadas:")
print(loader.data[feature_cols].describe())

In [ ]:
# Preparar secuencias para LSTM
X_train, X_test, y_train, y_test = loader.prepare_for_lstm(
    feature_cols,
    seq_length=config.SEQUENCE_LENGTH,
    split_ratio=config.TRAIN_TEST_SPLIT
)

print(f"\nFormas de datos:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test: {y_test.shape}")

## 3. Entrenamiento del Modelo LSTM

In [ ]:
# Crear y entrenar modelo LSTM
lstm_model = SeismicLSTM(seq_length=config.SEQUENCE_LENGTH, features=len(feature_cols))

history = lstm_model.train(
    X_train, y_train,
    X_val=X_test, y_val=y_test,
    epochs=50,
    batch_size=config.BATCH_SIZE,
    verbose=1
)

In [ ]:
# Evaluar modelo
metrics = lstm_model.evaluate(X_test, y_test)

## 4. Visualización de Resultados

In [ ]:
# Gráfica de pérdida
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida (MSE)')
axes[0].set_title('Pérdida durante Entrenamiento')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Predicciones vs Reales
y_pred = lstm_model.predict(X_test)
axes[1].plot(y_test[:200, 0], label='Valores Reales', alpha=0.7, linewidth=2)
axes[1].plot(y_pred[:200, 0], label='Predicciones', alpha=0.7, linewidth=2)
axes[1].set_xlabel('Muestras de Test')
axes[1].set_ylabel('Valor Normalizado')
axes[1].set_title('Predicciones vs Valores Reales')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Detección de Anomalías

In [ ]:
# Entrenar detector de anomalías
X_train_flat = X_train.reshape(X_train.shape[0], -1)
anomaly_detector = SeismicAnomalyDetector(contamination=0.1)
anomaly_detector.train(X_train_flat)

# Detectar anomalías en test
X_test_flat = X_test.reshape(X_test.shape[0], -1)
anomalies = anomaly_detector.predict(X_test_flat)
anomaly_scores = anomaly_detector.predict_proba(X_test_flat)

print(f"\nResultados de detección de anomalías:")
print(f"  Total de muestras: {len(anomalies)}")
print(f"  Anomalías detectadas: {(anomalies == -1).sum()}")
print(f"  Porcentaje: {(anomalies == -1).sum() / len(anomalies) * 100:.2f}%")

In [ ]:
# Visualizar scores de anomalía
plt.figure(figsize=(14, 4))
plt.scatter(range(len(anomaly_scores)), anomaly_scores, 
           c=anomalies, cmap='RdYlGn_r', s=30, alpha=0.6)
plt.axhline(y=0.5, color='r', linestyle='--', linewidth=2, label='Threshold')
plt.xlabel('Muestras')
plt.ylabel('Anomaly Score')
plt.title('Scores de Detección de Anomalías')
plt.colorbar(label='Clasificación (-1=Anomalía, 1=Normal)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Predicción en Tiempo Real

In [ ]:
# Simular predicción con últimos datos
X_recent = X_test[-1].reshape(1, config.SEQUENCE_LENGTH, len(feature_cols))

next_prediction = lstm_model.predict_single(X_recent)
next_anomaly_score = anomaly_detector.predict_proba(X_test_flat[-1].reshape(1, -1))[0]

print(f"\n" + "="*60)
print(f"PREDICCIÓN SIGUIENTE EVENTO")
print(f"="*60)
print(f"\nValor Predicho (normalizado): {next_prediction:.4f}")
print(f"Score de Anomalía: {next_anomaly_score:.4f}")

if next_anomaly_score < 0.3:
    print(f"Estado: 🟢 NORMAL - Actividad esperada")
elif next_anomaly_score < 0.6:
    print(f"Estado: 🟡 MODERADO - Comportamiento inusual")
elif next_anomaly_score < 0.8:
    print(f"Estado: 🟠 ALTO - Anomalía significativa")
else:
    print(f"Estado: 🔴 MUY ALTO - Comportamiento extremo")

print(f"\n" + "="*60)